In [210]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [211]:
df = pd.read_csv("train.txt" , sep = ";",header = None, names = ["text", "emotion"])

In [212]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [213]:
unique_emotions = df["emotion"].unique()
emotion_numbers = {}
i=0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i+=1

df["emotion"] = df["emotion"].map(emotion_numbers)

In [214]:
df["text"] = df["text"].apply(lambda x : x.lower())

In [215]:
import string

def remove_punc(txt):
    return txt.translate(str.maketrans("","",string.punctuation))

In [216]:
df["text"] = df["text"].apply(remove_punc)

In [217]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new
df["text"] = df["text"].apply(remove_numbers)

In [218]:
def remove_emojis(txt):
    new= ""
    for i in txt:
        if i.isascii():
            new = new+i
    return new
df["text"] = df["text"].apply(remove_emojis)
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [219]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize 

In [220]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Pcc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Pcc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Pcc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [221]:
stop_words = set(stopwords.words("english"))

In [222]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)

    return " ".join(cleaned)

In [223]:
df["text"] = df["text"].apply(remove)

In [224]:
df.loc[6]["text"]

'ive taking milligrams times recommended amount ive fallen asleep lot faster also feel like funny'

In [225]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    df["text"], df["emotion"], test_size = 0.2, random_state = 42
)

In [226]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [227]:
bow_vectorizer = CountVectorizer()

In [228]:
x_train_bow = bow_vectorizer.fit_transform(x_train)
x_test_bow = bow_vectorizer.transform(x_test)

NAIVE BAYES

In [229]:
nb_model = MultinomialNB()
nb_model.fit(x_train_bow, y_train)

y_pred = nb_model.predict(x_test_bow)
print("accuracy score : " , accuracy_score(y_test, y_pred))

accuracy score :  0.7678125


In [230]:
tfidf = TfidfVectorizer()
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [231]:
nb_model = MultinomialNB()
nb_model.fit(x_train_tfidf, y_train)

y_pred = nb_model.predict(x_test_tfidf)
print("accuracy score : " , accuracy_score(y_test, y_pred))

accuracy score :  0.6609375


LOGISTIC REGRESSION

In [232]:
from sklearn.linear_model import LogisticRegression

In [233]:
lr_model = LogisticRegression()
lr_model.fit(x_train_bow, y_train)

y_pred = lr_model.predict(x_test_bow)
print("accuracy score : " , accuracy_score(y_test, y_pred))

accuracy score :  0.88875


In [234]:
lr_model =  LogisticRegression()
lr_model.fit(x_train_tfidf, y_train)

y_pred = lr_model.predict(x_test_tfidf)
print("accuracy score : " , accuracy_score(y_test, y_pred))

accuracy score :  0.8615625
